## test_theirstack

In [29]:
import os
import json
import logging
from datetime import datetime, timezone
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

print("Imports OK")
print(f"THEIRSTACK_KEY loaded: {'yes' if os.getenv('THEIRSTACK_KEY') else '⚠️  NOT FOUND — check your .env file'}")

Imports OK
THEIRSTACK_KEY loaded: yes


In [30]:
BASE_URL = "https://api.theirstack.com/v1/jobs/search"
API_KEY  = os.getenv("THEIRSTACK_KEY")

HEADERS = {
    "Content-Type":  "application/json",
    "Authorization": f"Bearer {API_KEY}",
}

SEARCH_BODY = {
    "job_title_pattern_or": [
        "(?i)data analyst",
        "(?i)analytics engineer",
    ],
    "job_title_pattern_not": [
        "(?i)senior",
        "(?i)\\bsr\\b",
        "(?i)\\bii\\b",
        "(?i)\\biii\\b",
        "(?i)\\biv\\b",
        "(?i)lead",
        "(?i)principal",
        "(?i)staff",
        "(?i)manager",
        "(?i)director",
        "(?i)head of",
        "(?i)unpaid",
        "(?i)non.paid",
        "(?i)volunteer",
    ],
    "job_seniority_or":        ["junior", "mid_level"],
    "job_country_code_or":     ["US"],
    "job_location_pattern_or": [
        "(?i)^new york,",
        "(?i)new york city",
        "(?i)manhattan",
        "(?i)brooklyn",
    ],
    "posted_at_max_age_days": 7,
    "page":                   0,
}

PRODUCTION_LIMIT = 15   # max credits to spend per run

CACHE_DIR        = Path("./cache")
CACHE_DIR.mkdir(exist_ok=True)
FREE_SWEEP_CACHE = CACHE_DIR / "theirstack_free_sweep.json"
PAID_CACHE       = CACHE_DIR / "theirstack_paid_response.json"

print(f"Search body:\n{json.dumps(SEARCH_BODY, indent=2)}")

Search body:
{
  "job_title_pattern_or": [
    "(?i)data analyst",
    "(?i)analytics engineer"
  ],
  "job_title_pattern_not": [
    "(?i)senior",
    "(?i)\\bsr\\b",
    "(?i)\\bii\\b",
    "(?i)\\biii\\b",
    "(?i)\\biv\\b",
    "(?i)lead",
    "(?i)principal",
    "(?i)staff",
    "(?i)manager",
    "(?i)director",
    "(?i)head of",
    "(?i)unpaid",
    "(?i)non.paid",
    "(?i)volunteer"
  ],
  "job_seniority_or": [
    "junior",
    "mid_level"
  ],
  "job_country_code_or": [
    "US"
  ],
  "job_location_pattern_or": [
    "(?i)^new york,",
    "(?i)new york city",
    "(?i)manhattan",
    "(?i)brooklyn"
  ],
  "posted_at_max_age_days": 7,
  "page": 0
}


In [31]:
from collections import defaultdict

# ------------------------------------------------------------------ #
# Stage 1: Free sweep across all pages
# ------------------------------------------------------------------ #
if FREE_SWEEP_CACHE.exists():
    print("Free sweep cache exists — loading from file.")
    with FREE_SWEEP_CACHE.open() as f:
        all_free_jobs = json.load(f)
else:
    all_free_jobs = []
    page      = 0
    PAGE_SIZE = 25

    while True:
        sweep_body = {
            **SEARCH_BODY,
            "blur_company_data":     True,
            "include_total_results": page == 0,  # only need total on first page
            "limit":                 PAGE_SIZE,
            "page":                  page,
        }

        log.info(f"Free sweep page {page} — zero credits consumed.")
        response = requests.post(BASE_URL, json=sweep_body, headers=HEADERS, timeout=30)
        response.raise_for_status()
        data = response.json()

        if page == 0:
            total_available = data.get("metadata", {}).get("total_results", "?")
            log.info(f"Total matching jobs in TheirStack: {total_available}")

        page_jobs = data.get("data", [])
        if not page_jobs:
            log.info("Empty page — sweep complete.")
            break

        all_free_jobs.extend(page_jobs)
        log.info(f"  Page {page}: {len(page_jobs)} jobs "
                 f"(running total: {len(all_free_jobs)})")

        if len(page_jobs) < PAGE_SIZE:
            log.info("Partial page — reached last page.")
            break

        page += 1

    with FREE_SWEEP_CACHE.open("w") as f:
        json.dump(all_free_jobs, f, indent=2)
    log.info(f"Free sweep saved to {FREE_SWEEP_CACHE}")

# ------------------------------------------------------------------ #
# Stage 2: Display all results
# ------------------------------------------------------------------ #
print(f"\nTotal jobs collected : {len(all_free_jobs)}")
print()
print(f"{'#':<4} {'Job Title':<55} {'Seniority':<12} {'Location':<20} "
      f"{'Posted':<12} {'Technologies'}")
print("-" * 140)
for i, job in enumerate(all_free_jobs):
    print(
        f"{i:<4} "
        f"{job.get('job_title', 'N/A'):<55} "
        f"{job.get('seniority', 'N/A'):<12} "
        f"{(job.get('short_location') or job.get('location') or 'N/A'):<20} "
        f"{(job.get('date_posted') or 'N/A'):<12} "
        f"{', '.join(job.get('technology_slugs', [])[:5])}"
    )

# ------------------------------------------------------------------ #
# Stage 3: Deduplicate
# ------------------------------------------------------------------ #
# Fingerprint logic:
# Same job posted on multiple boards will be parsed by TheirStack's
# algorithm identically, producing the same title, location string,
# and technology slug set. We rely on this determinism rather than
# fuzzy matching. Edge case: same job with "New York" on one board
# and "New York, NY" on another will slip through — acceptable
# tradeoff given how rare this is vs the complexity of handling it.
print("\n" + "=" * 60)
print("DEDUPLICATION")
print("=" * 60)

fingerprint_groups = defaultdict(list)
for job in all_free_jobs:
    fingerprint = (
        job.get("job_title", "").lower().strip(),
        job.get("short_location", "").lower().strip(),
        frozenset(job.get("technology_slugs", [])),
    )
    fingerprint_groups[fingerprint].append(job)

unique_jobs = [group[0] for group in fingerprint_groups.values()]
dupe_groups = {k: v for k, v in fingerprint_groups.items() if len(v) > 1}

print(f"\nTotal records        : {len(all_free_jobs)}")
print(f"Unique after dedup   : {len(unique_jobs)}")
print(f"Duplicate groups     : {len(dupe_groups)}")

if dupe_groups:
    print("\n=== Duplicate groups ===")
    
    # Build a lookup of id -> original index for quick reference
    id_to_index = {job["id"]: i for i, job in enumerate(all_free_jobs)}
    
    for (title, location, techs), group in dupe_groups.items():
        print(f"\n  Title    : {title}")
        print(f"  Location : {location}")
        print(f"  Techs    : {sorted(techs)[:5]}")
        print(f"  Copies   : {len(group)}")
        for j in group:
            original_index = id_to_index.get(j["id"], "?")
            print(f"    #{original_index:<4} id={j['id']}  "
                  f"posted={j.get('date_posted')!r}  "
                  f"source_url={j.get('source_url', '')[:70]}")

# ------------------------------------------------------------------ #
# Stage 4: Build unique ID list for paid fetch
# ------------------------------------------------------------------ #
all_unique_ids   = [job["id"] for job in unique_jobs]
production_ids   = all_unique_ids[:PRODUCTION_LIMIT]

print(f"\n{'=' * 60}")
print(f"PRODUCTION ID LIST")
print(f"{'=' * 60}")
print(f"Total unique jobs    : {len(all_unique_ids)}")
print(f"IDs for paid fetch   : {len(production_ids)} "
      f"(capped at PRODUCTION_LIMIT={PRODUCTION_LIMIT})")
print(f"Credits this run     : {len(production_ids)}")
print(f"\nIDs: {production_ids}")

2026-05-23 14:33:42,803 [INFO] Free sweep page 0 — zero credits consumed.
2026-05-23 14:33:48,872 [INFO] Total matching jobs in TheirStack: 45
2026-05-23 14:33:48,875 [INFO]   Page 0: 25 jobs (running total: 25)
2026-05-23 14:33:48,876 [INFO] Free sweep page 1 — zero credits consumed.
2026-05-23 14:33:53,292 [INFO]   Page 1: 20 jobs (running total: 45)
2026-05-23 14:33:53,293 [INFO] Partial page — reached last page.
2026-05-23 14:33:53,335 [INFO] Free sweep saved to cache/theirstack_free_sweep.json



Total jobs collected : 45

#    Job Title                                               Seniority    Location             Posted       Technologies
--------------------------------------------------------------------------------------------------------------------------------------------
0    Data Analyst                                            mid_level    New York             2026-05-23   microsoft-excel, data-studio, tableau, power-bi
1    Junior Data Analyst                                     junior       New York             2026-05-22   microsoft-excel, tableau, power-bi, python
2    Data Analyst - Energy Programs | W2 Only (No C2C / No Visa) mid_level    New York, NY         2026-05-22   built-in, salesforce, microsoft-power-platform, power-bi, microsoft-power-apps
3    Data Analyst                                            mid_level    New York, NY         2026-05-22   salesforce, salesforceorg, the-org
4    Data Analyst - Energy Programs | W2 Only (No C2C / No Visa) mid_

In [32]:
if PAID_CACHE.exists():
    print("Paid cache exists — loading from file.")
    with PAID_CACHE.open() as f:
        paid_response = json.load(f)
else:
    if not production_ids:
        raise ValueError("production_ids is empty — run Cell 3 first.")

    paid_body = {
        "job_id_or":             production_ids,
        "blur_company_data":     False,
        "include_total_results": False,
        "limit":                 PRODUCTION_LIMIT,
    }

    log.info(f"Firing paid request — {len(production_ids)} credits consumed.")
    response = requests.post(BASE_URL, json=paid_body, headers=HEADERS, timeout=30)
    response.raise_for_status()
    paid_response = response.json()

    with PAID_CACHE.open("w") as f:
        json.dump(paid_response, f, indent=2)
    log.info("Paid response saved.")

paid_jobs   = paid_response.get("data", [])
ingested_at = datetime.now(timezone.utc).isoformat()

snowflake_rows = [
    {
        "SOURCE":      "theirstack",
        "RAW_PAYLOAD": job,
        "INGESTED_AT": ingested_at,
    }
    for job in paid_jobs
]

print(f"Jobs returned        : {len(paid_jobs)}")
print(f"Snowflake rows ready : {len(snowflake_rows)}")
print()

# Confirm unblurred
print("Spot check — company names should be visible:")
for job in paid_jobs[:3]:
    print(f"  {job.get('job_title',''):<50} @ {job.get('company',''):<30} "
          f"| {job.get('short_location','')}")

print()
print("Sample Snowflake row (key fields):")
print(json.dumps(
    {
        "SOURCE":      snowflake_rows[0]["SOURCE"],
        "INGESTED_AT": snowflake_rows[0]["INGESTED_AT"],
        "RAW_PAYLOAD": {
            k: v for k, v in snowflake_rows[0]["RAW_PAYLOAD"].items()
            if k in ["id", "job_title", "company", "location",
                     "date_posted", "discovered_at", "seniority",
                     "technology_slugs", "salary_string"]
        },
    },
    indent=2,
    default=str,
))

2026-05-23 14:35:41,897 [INFO] Firing paid request — 15 credits consumed.
2026-05-23 14:35:51,560 [INFO] Paid response saved.


Jobs returned        : 15
Snowflake rows ready : 15

Spot check — company names should be visible:
  Data Analyst                                       @ Aisle and Abroad               | New York
  Junior Data Analyst                                @ Nova Infotek                   | New York
  Data Analyst - Energy Programs | W2 Only (No C2C / No Visa) @ Amerit Consulting              | New York, NY

Sample Snowflake row (key fields):
{
  "SOURCE": "theirstack:nyc-data-roles",
  "INGESTED_AT": "2026-05-23T18:35:51.561826+00:00",
  "RAW_PAYLOAD": {
    "id": 691778902,
    "job_title": "Data Analyst",
    "date_posted": "2026-05-23",
    "company": "Aisle and Abroad",
    "location": "New York",
    "salary_string": null,
    "seniority": "mid_level",
    "discovered_at": "2026-05-23T12:00:03.057000Z",
    "technology_slugs": [
      "microsoft-excel",
      "data-studio",
      "tableau",
      "power-bi"
    ]
  }
}


In [3]:
import json
import requests
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("THEIRSTACK_KEY")

response = requests.get(
    "https://api.theirstack.com/v0/billing/credit-balance",
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    },
    timeout=15,
)

print(f"HTTP status: {response.status_code}")
print()

if response.status_code == 200:
    data = response.json()
    print("Full response:")
    print(json.dumps(data, indent=2))
else:
    print(f"❌ Failed: {response.text}")

HTTP status: 200

Full response:
{
  "ui_credits": 50,
  "used_ui_credits": 0,
  "api_credits": 200,
  "used_api_credits": 45,
  "earliest_expiration": "2026-06-21T13:30:40.501000Z"
}
